In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install transformers torch scikit-learn accelerate

Cell 2 we check if our colab has GPU, If it prints CUDA then we can carry on. If it prints cpu, we have to change it to GPU in runtime.

In [4]:
import os
import torch
import numpy as np
from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score
)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


3.We label map, Afrisenti uses these three classes we will use.We map them to numbers because this model uses integers not strings.

In [5]:
PROJECT_PATH = '/content/drive/Shareddrives/Cos760'


LBL2ID = {'negative': 0, 'neutral': 1, 'positive': 2}
ID2LBL = {0: 'negative', 1: 'neutral', 2: 'positive'}


MODEL_NAME = "xlm-roberta-base"

LANGUAGES = ['hausa', 'kinyarwanda']

4.The tokenizer is loaded from HuggingFace using the same configuration that XLM-R was originally pretrained with, ensuring that text is broken up in exactly the same way the model expects. The tokenize function converts each tweet from raw text into token IDs the model can process, padding shorter tweets to a fixed length of 128 and truncating anything longer. The encode_labels function converts the text sentiment labels positive, negative, and neutral into integers 2, 0, and 1 respectively, since the model requires numerical inputs for training

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch['tweet'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

def encode_labels(example):
    example['label'] = LBL2ID[example['label']]
    return example

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

5.cell 5 we define our metrics

In [7]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        'f1': f1_score(labels, predictions, average='weighted'),
        'precision': precision_score(labels, predictions, average='weighted', zero_division=0),
        'recall': recall_score(labels, predictions, average='weighted', zero_division=0),
        'accuracy': accuracy_score(labels, predictions)
    }


6.This cell defines a custom SentimentDataset class that loads tweets and labels from the processed CSV files saved in notebook 01 and converts them into a format PyTorch can use for training. The class handles tokenization and label encoding for each tweet individually when the model requests it. The training loop then iterates over both Hausa and Kinyarwanda, loading the train, validation, and test splits for each language and wrapping them in the dataset class. XLM-R is loaded with a three-class classification head and fine-tuned using the HuggingFace Trainer with a learning rate of 2e-5, batch size of 16, and up to 5 epochs. Early stopping is applied with a patience of 2, meaning training stops automatically if the weighted F1 score on the validation set does not improve for two consecutive epochs.

In [8]:
import pandas as pd
from torch.utils.data import Dataset

PROJECT_PATH = '/content/drive/Shareddrives/Cos760'

LBL2ID = {'negative': 0, 'neutral': 1, 'positive': 2}

# Custom Dataset class to load from CSV
class SentimentDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=128):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tweet = str(self.data.iloc[idx]['cleaned_tweet'])
        label = LBL2ID[self.data.iloc[idx]['label']]

        encoding = self.tokenizer(
            tweet,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

for lang in LANGUAGES:
    print(f"\n{'='*60}")
    print(f"Fine-tuning XLM-R on {lang.upper()}")
    print(f"{'='*60}")

    # Load CSVs
    train_df = pd.read_csv(os.path.join(PROJECT_PATH, 'data/processed', f'{lang}_train_cleaned.csv'))
    val_df   = pd.read_csv(os.path.join(PROJECT_PATH, 'data/processed', f'{lang}_validation_cleaned.csv'))
    test_df  = pd.read_csv(os.path.join(PROJECT_PATH, 'data/processed', f'{lang}_test_cleaned.csv'))

    print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

    # Create datasets
    train_dataset = SentimentDataset(train_df, tokenizer)
    val_dataset   = SentimentDataset(val_df, tokenizer)
    test_dataset  = SentimentDataset(test_df, tokenizer)

    # Load model
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=3,
        id2label=ID2LBL,
        label2id=LBL2ID
    ).to(device)

    # Training settings
    training_args = TrainingArguments(
        output_dir=os.path.join(PROJECT_PATH, f'models/xlmr/{lang}'),
        num_train_epochs=5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        warmup_steps=100,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        logging_dir=os.path.join(PROJECT_PATH, f'outputs/metrics/xlmr_{lang}'),
        logging_steps=50,
        fp16=torch.cuda.is_available()
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()

    print(f"\nTest set results for {lang}:")
    results = trainer.evaluate(test_dataset)
    print(results)

    model.save_pretrained(os.path.join(PROJECT_PATH, f'models/xlmr/{lang}/final'))
    tokenizer.save_pretrained(os.path.join(PROJECT_PATH, f'models/xlmr/{lang}/final'))
    print(f"\n XLM-R fine-tuned and saved for {lang}")


Fine-tuning XLM-R on HAUSA
Train: 14172 | Val: 2677 | Test: 5303


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,F1,Precision,Recall,Accuracy
1,0.748551,0.699561,0.688795,0.711360,0.689204,0.689204
2,0.603377,0.667426,0.727417,0.736620,0.731042,0.731042
3,0.567261,0.601404,0.754397,0.755114,0.753829,0.753829
4,0.461882,0.638603,0.752467,0.751991,0.753082,0.753082
5,0.397128,0.673908,0.754833,0.755642,0.754576,0.754576


...

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Test set results for kinyarwanda:


{'eval_loss': 0.9754760265350342, 'eval_f1': 0.5265610086076812, 'eval_precision': 0.5507237212577432, 'eval_recall': 0.5399610136452242, 'eval_accuracy': 0.5399610136452242, 'eval_runtime': 2.3116, 'eval_samples_per_second': 443.846, 'eval_steps_per_second': 14.276, 'epoch': 5.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


 XLM-R fine-tuned and saved for kinyarwanda


7. This cell repeats the same fine-tuning process as XLM-R but using AfriBERTa, a model pretrained specifically on 11 African languages. AfriBERTa has its own tokenizer which is loaded separately since it was trained on a different vocabulary. The same hyperparameter settings are applied to both models to ensure a fair comparison, meaning any performance differences can be attributed to pretraining data composition rather than training configuration.

In [9]:

AFRIBERTA_MODEL = "castorini/afriberta_large"

for lang in LANGUAGES:
    print(f"\n{'='*60}")
    print(f"Fine-tuning AfriBERTa on {lang.upper()}")
    print(f"{'='*60}")

    # Load tokenizer for AfriBERTa
    afriberta_tokenizer = AutoTokenizer.from_pretrained(AFRIBERTA_MODEL)

    # Load CSVs
    train_df = pd.read_csv(os.path.join(PROJECT_PATH, 'data/processed', f'{lang}_train_cleaned.csv'))
    val_df   = pd.read_csv(os.path.join(PROJECT_PATH, 'data/processed', f'{lang}_validation_cleaned.csv'))
    test_df  = pd.read_csv(os.path.join(PROJECT_PATH, 'data/processed', f'{lang}_test_cleaned.csv'))

    print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

    # Create datasets using AfriBERTa tokenizer
    train_dataset = SentimentDataset(train_df, afriberta_tokenizer)
    val_dataset   = SentimentDataset(val_df, afriberta_tokenizer)
    test_dataset  = SentimentDataset(test_df, afriberta_tokenizer)

    # Load AfriBERTa with classification head
    model = AutoModelForSequenceClassification.from_pretrained(
        AFRIBERTA_MODEL,
        num_labels=3,
        id2label=ID2LBL,
        label2id=LBL2ID
    ).to(device)

    # Same training settings as XLM-R for fair comparison
    training_args = TrainingArguments(
        output_dir=os.path.join(PROJECT_PATH, f'models/afriberta/{lang}'),
        num_train_epochs=5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        warmup_steps=100,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        logging_dir=os.path.join(PROJECT_PATH, f'outputs/metrics/afriberta_{lang}'),
        logging_steps=50,
        fp16=torch.cuda.is_available()
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()

    print(f"\nTest set results for {lang}:")
    results = trainer.evaluate(test_dataset)
    print(results)

    model.save_pretrained(os.path.join(PROJECT_PATH, f'models/afriberta/{lang}/final'))
    afriberta_tokenizer.save_pretrained(os.path.join(PROJECT_PATH, f'models/afriberta/{lang}/final'))
    print(f"\n AfriBERTa fine-tuned and saved for {lang}")


Fine-tuning AfriBERTa on HAUSA


config.json:   0%|          | 0.00/731 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/257 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/1.55M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

...

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Test set results for kinyarwanda:


{'eval_loss': 1.1938971281051636, 'eval_f1': 0.6193261088382385, 'eval_precision': 0.6192234914251821, 'eval_recall': 0.6198830409356725, 'eval_accuracy': 0.6198830409356725, 'eval_runtime': 1.9772, 'eval_samples_per_second': 518.928, 'eval_steps_per_second': 16.691, 'epoch': 5.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


 AfriBERTa fine-tuned and saved for kinyarwanda
